# Representation Studies — t-SNE Class Separation

Pipeline:
1. **Load the model** from a provided directory (config + weights).
2. **Extract Transformer representations** (pre-classifier features) on Capture-24 data.
3. **t-SNE visualization** (scikit-learn) to analyze class separation.

In [5]:
# Ensure project root (folder containing capture24) is on path
import os
import sys
_root = os.path.abspath(os.getcwd())
while _root and not os.path.isdir(os.path.join(_root, 'capture24')):
    _parent = os.path.dirname(_root)
    if _parent == _root:
        break
    _root = _parent
if _root and _root not in sys.path:
    sys.path.insert(0, _root)

os.environ["WANDB_MODE"] = "disabled"

import torch
import numpy as np
import matplotlib.pyplot as plt
import yaml
import json
import gzip
from torch.utils.data import DataLoader
from sklearn.manifold import TSNE

from models.patch_tst import PatchTST

## 1. Configuration — Model and Data Paths

Set `MODEL_DIR` to the directory containing the saved model (e.g. `config.yaml` + `patchtst_model.pth`, or `checkpoint.pt`).  
Set `DATA_DIR` to the Capture-24 data folder (e.g. `capture24/final_data_2048/`) with `X_test.npy.gz`, `Y_test.npy.gz`, `label_to_index.json`.

In [6]:
MODEL_DIR = 'pl128_s16_lr0.001_do0.2_L5_optadamw_h8_wd0p03/'
DATA_DIR = 'capture24/final_data_1024_mode_v/'

# Optional: subsample for faster t-SNE (None = use all test samples)
MAX_SAMPLES = 2000

# t-SNE parameters
TSNE_PERPLEXITY = 30
TSNE_RANDOM_STATE = 42

## 2. Load Model from Directory

Supports either:
- `config.yaml` + `patchtst_model.pth` in `MODEL_DIR`
- Single `checkpoint.pt` (contains config and model_state_dict)

In [14]:
def load_model_from_dir(model_dir, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
    model_dir = model_dir.rstrip('/')
    config_path = os.path.join(model_dir, 'config.yaml')
    checkpoint_path = os.path.join(model_dir, 'checkpoint.pt')
    state_path = os.path.join(model_dir, 'patchtst_model.pth')

    if os.path.isfile(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        config = ckpt['config']
        state_dict = ckpt['model_state_dict']
    elif os.path.isfile(config_path) and os.path.isfile(state_path):
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
        state_dict = torch.load(state_path, map_location=device, weights_only=True)
    else:
        raise FileNotFoundError(
            f"Model not found in {model_dir}. "
            "Expect either checkpoint.pt or (config.yaml + patchtst_model.pth)."
        )

    config['hook_attention_maps'] = False
    config.setdefault('positional_encoding', 'fixed')
    config.setdefault('classifier', 'linear')
    # Match architecture to checkpoint: learned table if state_dict has it
    if 'pos_encoding' in state_dict:
        config['positional_encoding'] = 'learned'
    model = PatchTST(config)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model, config, device

In [15]:
model, config, device = load_model_from_dir(MODEL_DIR)
print(f"Loaded model from {MODEL_DIR}")
print(f"Device: {device}")
print(f"Lookback: {config.get('lookback_window')}, num_classes: {config.get('num_classes')}")

Loaded model from pl128_s16_lr0.001_do0.2_L5_optadamw_h8_wd0p03/
Device: mps
Lookback: 1024, num_classes: 10


## 3. Load Capture-24 Test Data Only

We load **only the test split** (X_test, Y_test) from Capture-24. No train or validation data is used; the t-SNE plot is computed solely on this test set.

In [9]:
# Load only the test split (no train/validation)
with gzip.open(f'{DATA_DIR}/X_test.npy.gz', 'rb') as f:
    X_test = np.load(f)
with gzip.open(f'{DATA_DIR}/Y_test.npy.gz', 'rb') as f:
    Y_test = np.load(f)
with open(f'{DATA_DIR}/label_to_index.json', 'r') as f:
    label_data = json.load(f)

idx_to_label = label_data['index_to_label']
label_to_idx = label_data['label_to_index']
class_names = list(label_to_idx.keys())

if MAX_SAMPLES is not None and len(X_test) > MAX_SAMPLES:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(X_test), size=MAX_SAMPLES, replace=False)
    X_test = X_test[idx]
    Y_test = Y_test[idx]

print(f"Capture-24 test samples (plot uses only these): {len(X_test)}, shape: {X_test.shape}")
print(f"Classes: {class_names}")

Capture-24 test samples (plot uses only these): 2000, shape: (2000, 1024, 3)
Classes: ['bicycling', 'household-chores', 'manual-work', 'mixed-activity', 'sitting', 'sleep', 'sports', 'standing', 'vehicle', 'walking']


## 4. Extract Transformer Representations

We capture the **pre-classifier** representation (output of the encoder, flattened). A forward hook on the classifier input is used to extract these features.

In [10]:
def extract_representations(model, X, device, batch_size=128):
    """Extract (B, D) representation for each sample using a hook on classifier input."""
    collected = []

    def hook_fn(module, input, output):
        collected.append(input[0].detach())

    handle = model.classifier.register_forward_hook(hook_fn)
    dataset = torch.utils.data.TensorDataset(torch.from_numpy(X).float())
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for (x_batch,) in loader:
            x_batch = x_batch.to(device)
            _ = model(x_batch)
    handle.remove()

    return torch.cat(collected, dim=0).cpu().numpy()

In [11]:
features = extract_representations(model, X_test, device, batch_size=config.get('num_batches', 128))
labels = Y_test.astype(int)
print(f"Representations shape: {features.shape}")
print(f"Labels shape: {labels.shape}")

AttributeError: 'Parameter' object has no attribute 'forward'

## 5. t-SNE and Class-Separation Plot

Run t-SNE on the extracted representations and plot points colored by activity class.

In [ ]:
tsne = TSNE(n_components=2, perplexity=TSNE_PERPLEXITY, random_state=TSNE_RANDOM_STATE, n_iter=1000)
embeddings_2d = tsne.fit_transform(features)
print("t-SNE fit complete.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
unique_labels = np.unique(labels)
colors = plt.cm.tab10(np.linspace(0, 1, max(10, len(unique_labels))))

for i, label_id in enumerate(unique_labels):
    mask = labels == label_id
    name = idx_to_label.get(str(label_id), str(label_id))
    ax.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=[colors[i]],
        label=name,
        alpha=0.6,
        s=15,
        edgecolors='none',
    )

ax.set_title('t-SNE of PatchTST representations (Capture-24 test set only)')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()